In [2]:
# Import required libraries
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression

# =========================
# 1. LOAD DATA
# =========================
# Read CSV file
df = pd.read_csv("data.csv")

# Keep only required columns
df = df[["Restaurant Name", "Cuisines"]]

# Remove rows where Restaurant Name or Cuisines are empty
df = df.dropna(subset=["Cuisines", "Restaurant Name"])

# =========================
# 2. CLEAN CUISINES COLUMN
# =========================
# Convert "Burger, Fast Food" → ["Burger", "Fast Food"]
def fix_cuisines(x):
    if isinstance(x, str):
        return [c.strip() for c in x.split(",")]
    return []

df["Cuisines"] = df["Cuisines"].apply(fix_cuisines)

# Remove rows where cuisine list is empty
df = df[df["Cuisines"].map(len) > 0]

# (Optional checks)
df["Cuisines"].head()
df["Cuisines"].iloc[0]

# =========================
# 3. REMOVE VERY RARE CUISINES
# =========================
# Count how many times each cuisine appears
cuisine_counts = Counter(
    c for cuisines in df["Cuisines"] for c in cuisines
)

# Keep cuisines that appear at least 20 times
valid_cuisines = {c for c, cnt in cuisine_counts.items() if cnt >= 20}

# Remove rare cuisines from each row
df["Cuisines"] = df["Cuisines"].apply(
    lambda x: [c for c in x if c in valid_cuisines]
)

# Remove rows that became empty after filtering
df = df[df["Cuisines"].map(len) > 0]

# =========================
# 4. TEXT VECTORIZATION (NLP)
# =========================
# Convert restaurant names into numerical vectors using TF-IDF
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),      # use single words + word pairs
    min_df=2,               # ignore words appearing < 2 times
    stop_words="english"    # remove common English words
)

# Transform restaurant names into numbers
X = vectorizer.fit_transform(df["Restaurant Name"])

# =========================
# 5. MULTI-LABEL ENCODING
# =========================
# Convert cuisine lists into binary format
mlb = MultiLabelBinarizer()
y = mlb.fit_transform(df["Cuisines"])

# =========================
# 6. TRAIN-TEST SPLIT
# =========================
# Split data into training (80%) and testing (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# =========================
# 7. TRAIN MODEL
# =========================
# One-vs-Rest Logistic Regression for multi-label classification
model = OneVsRestClassifier(
    LogisticRegression(max_iter=300)
)

# Train the model
model.fit(X_train, y_train)

# =========================
# 8. USER INPUT
# =========================
# Take restaurant name from user
restaurant_name = input("Enter Restaurant Name: ")

# Convert input text into TF-IDF vector
X_input = vectorizer.transform([restaurant_name])

# Predict probabilities for each cuisine
probs = model.predict_proba(X_input)[0]

# =========================
# 9. FILTER PREDICTIONS
# =========================
# Select cuisines whose probability is above threshold
threshold = 0.2
predicted_indices = np.where(probs >= threshold)[0]

# Convert indices to cuisine names
predicted_cuisines = [mlb.classes_[i] for i in predicted_indices]

# =========================
# 10. OUTPUT
# =========================
# Print only cuisine names
print("Predicted Cuisines:", predicted_cuisines[0])


Enter Restaurant Name:  Gero


Predicted Cuisines: Chinese
